# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [67]:
print("Hello World!")


Hello World!


In [68]:
import pandas as pd

# Path to the file the teacher gave you
file_path = 'loinc_dataset-v2.xlsx'

# 1. Load the three sheets (skipping the query header at the top)
df_glucose = pd.read_excel(file_path, sheet_name='glucose in blood', skiprows=2)
df_bilirubin = pd.read_excel(file_path, sheet_name='bilirubin in plasma', skiprows=2)
df_wbc = pd.read_excel(file_path, sheet_name='White blood cells count', skiprows=2)

# 2. Assign Query IDs (qid) and Relevance Labels
# We tell the model which LOINC code is the "Perfect Match" for each query
def prepare_data(df, qid, winner_code):
    df['qid'] = qid
    df['relevance'] = 0  # Default all to 'wrong'
    # Mark the correct LOINC code as '1' (Correct)
    df.loc[df['loinc_num'] == winner_code, 'relevance'] = 1
    return df

# Using the most accurate codes found in your data
df_glucose = prepare_data(df_glucose, qid=1, winner_code='74774-1') # Glucose Bld
df_bilirubin = prepare_data(df_bilirubin, qid=2, winner_code='1975-2') # Bilirubin Plas
df_wbc = prepare_data(df_wbc, qid=3, winner_code='26464-8') # WBC Count

# 3. Merge them into one Master Training Set
training_set = pd.concat([df_glucose, df_bilirubin, df_wbc], ignore_index=True)

print("Training set built successfully!")
print(training_set[['qid', 'loinc_num', 'relevance']].head())

# Λεξικό με τις ερωτήσεις για σύγκριση
query_texts = {
    1: "glucose in blood",
    2: "bilirubin in plasma",
    3: "white blood cells count"
}

queries = {1: "glucose in blood", 2: "bilirubin in plasma", 3: "white blood cells count"}

def extract_features(row):
    query = queries[row['qid']].lower()
    name = str(row['long_common_name']).lower()

    # Feature: Πόσες λέξεις της ερώτησης υπάρχουν στο όνομα;
    match_count = sum(1 for word in query.split() if word in name)
    return match_count

training_set['feature_match'] = training_set.apply(extract_features, axis=1)

Training set built successfully!
   qid loinc_num  relevance
0    1    1988-5          0
1    1    1959-6          0
2    1   10331-7          0
3    1   18998-5          0
4    1    1975-2          0


In [69]:
from sklearn.ensemble import RandomForestClassifier

# X = τα χαρακτηριστικά, y = ο στόχος (σωστό/λάθος)
X = training_set[['feature_match']]
y = training_set['relevance']

model = RandomForestClassifier()
model.fit(X, y)

# Προβλέπουμε την πιθανότητα να είναι σωστό κάθε αποτέλεσμα
training_set['score'] = model.predict_proba(X)[:, 1]

In [70]:
# Ταξινομούμε το training_set ανά qid (ερώτηση) και μετά ανά score (πιθανότητα)
# Οι πιο "πιθανοί" κωδικοί θα πάνε στην κορυφή
final_results = training_set.sort_values(by=['qid', 'score'], ascending=[True, False])

# Εμφάνιση των 5 πρώτων αποτελεσμάτων για κάθε ερώτηση
for qid in queries.keys():
    print(f"\n--- Top 5 Results for Query {qid}: {queries[qid]} ---")
    top_5 = final_results[final_results['qid'] == qid].head(5)
    print(top_5[['loinc_num', 'long_common_name', 'score', 'relevance']])


--- Top 5 Results for Query 1: glucose in blood ---
   loinc_num                                   long_common_name     score  \
23   74774-1    Glucose [Mass/volume] in Serum, Plasma or Blood  0.394957   
0     1988-5  C reactive protein [Mass/volume] in Serum or P...  0.009425   
4     1975-2   Bilirubin.total [Mass/volume] in Serum or Plasma  0.009425   
7    18906-8                     Ciprofloxacin [Susceptibility]  0.009425   
8     2143-6          Cortisol [Mass/volume] in Serum or Plasma  0.009425   

    relevance  
23          1  
0           0  
4           0  
7           0  
8           0  

--- Top 5 Results for Query 2: bilirubin in plasma ---
   loinc_num                                   long_common_name     score  \
76    1968-7  Bilirubin.direct [Mass/volume] in Serum or Plasma  0.394957   
81    1975-2   Bilirubin.total [Mass/volume] in Serum or Plasma  0.394957   
95    1971-1  Bilirubin.indirect [Mass/volume] in Serum or P...  0.394957   
99   35192-4  Bilirubin.

In [71]:
# Παίρνουμε 20 τυχαία δείγματα από όλο το training_set
extra_negatives = training_set.sample(n=20).copy()

# Τους αλλάζουμε το qid (π.χ. τα κάνουμε όλα να ανήκουν στην ερώτηση 1)
# και βάζουμε relevance 0
extra_negatives['qid'] = 1
extra_negatives['relevance'] = 0

# Επαναϋπολογίζουμε τα features για αυτά τα "νέα" δεδομένα
extra_negatives['feature_match'] = extra_negatives.apply(extract_features, axis=1)

# Τα προσθέτουμε στο αρχικό μας training_set
training_set_extended = pd.concat([training_set, extra_negatives], ignore_index=True)

print(f"Το dataset επεκτάθηκε! Νέο μέγεθος: {len(training_set_extended)} γραμμές.")

Το dataset επεκτάθηκε! Νέο μέγεθος: 221 γραμμές.
